# Neo4j를 활용한 GraphRAG (Graph Retrieval-Augmented Generation)

## 1. RAG (Retrieval-Augmented Generation)란?

### 1.1 RAG의 필요성

**문제점**: 대규모 언어모델(LLM)의 한계
- ❌ **지식의 한계**: 학습 데이터 시점까지의 정보만 알고 있음
- ❌ **환각(Hallucination)**: 없는 정보를 만들어내는 경향
- ❌ **도메인 특화 지식 부족**: 특정 기업/조직의 내부 정보 모름
- ❌ **업데이트 어려움**: 새로운 정보를 반영하려면 재학습 필요

**해결책**: RAG (Retrieval-Augmented Generation)
- ✅ **최신 정보 제공**: 외부 데이터베이스에서 관련 정보 검색
- ✅ **정확성 향상**: 검색된 실제 데이터 기반 답변 생성
- ✅ **출처 제공**: 답변의 근거를 명확히 제시
- ✅ **비용 효율적**: 전체 모델 재학습 없이 정보 업데이트


### 1.2 RAG의 기본 구조

RAG는 크게 3단계로 구성됩니다:

```
┌─────────────┐
│  1. Index   │  문서를 벡터로 변환하여 DB에 저장
└──────┬──────┘
       │
       ↓
┌─────────────┐
│ 2. Retrieve │  질문과 유사한 문서를 검색
└──────┬──────┘
       │
       ↓
┌─────────────┐
│ 3. Generate │  LLM이 검색된 문서를 참고하여 답변 생성
└─────────────┘
```

**단계별 상세**:

1. **Index (색인)**: 
   - 문서를 작은 청크(chunk)로 분할
   - 임베딩 모델로 벡터화
   - 데이터베이스에 저장

2. **Retrieve (검색)**:
   - 사용자 질문을 벡터화
   - 유사도 기반으로 관련 문서 검색
   - Top-K 개의 가장 관련있는 문서 추출

3. **Generate (생성)**:
   - 검색된 문서를 컨텍스트로 제공
   - LLM이 질문 + 컨텍스트를 받아 답변 생성
   - 근거가 있는 정확한 답변 제공


## 2. VectorDB RAG vs GraphDB RAG 비교

### 2.1 Vector RAG (전통적인 RAG)

**데이터 저장 방식**: 벡터 데이터베이스 (예: Pinecone, Chroma, FAISS)

```
문서 A → [0.2, 0.8, 0.1, ...] ───┐
문서 B → [0.5, 0.3, 0.9, ...] ───┤ 벡터 DB
문서 C → [0.1, 0.7, 0.4, ...] ───┘
```

**장점** ✅:
- 의미적 유사도 검색에 강함
- 간단한 구조로 빠른 구현 가능
- 대용량 데이터 처리에 효율적
- 임베딩 기반 검색 성능 우수

**단점** ❌:
- **관계 정보 손실**: 엔티티 간 연결 관계를 파악하기 어려움
- **맥락 파악 한계**: "누가 누구와 어떤 관계인가?" 같은 질문에 약함
- **복잡한 추론 어려움**: 여러 단계의 관계 추론이 필요한 질문에 한계
- **중복 정보**: 같은 엔티티가 여러 문서에 분산되어 저장

**적합한 사용 사례**:
- 단순 키워드/의미 검색
- FAQ 시스템
- 문서 유사도 검색


### 2.2 Graph RAG (그래프 기반 RAG)

**데이터 저장 방식**: 그래프 데이터베이스 (예: Neo4j, Amazon Neptune)

```
    (기자A)──[WORKS_FOR]──→(언론사X)
       │                         ↑
       │                         │
   [WRITTEN_BY]            [PUBLISHED_BY]
       │                         │
       ↓                         │
    (뉴스1)──[BELONGS_TO]──→(카테고리)
       │
   [PUBLISHED_IN]
       │
       ↓
   (2025-11)
```

**장점** ✅:
- **관계 보존**: 엔티티 간의 연결 관계를 명시적으로 저장
- **복잡한 질의**: "이 기자가 작성한 경제 뉴스는?" 같은 복잡한 질문 처리
- **맥락 이해**: 그래프 경로를 따라가며 풍부한 맥락 제공
- **추론 가능**: 여러 홉(hop)을 거쳐 관계 추론
- **중복 제거**: 같은 엔티티는 하나의 노드로 통합

**단점** ❌:
- 초기 그래프 구축이 복잡함
- 벡터 검색보다 구현이 어려움
- 그래프 설계가 성능에 큰 영향

**적합한 사용 사례**:
- 지식 그래프 기반 QA
- 복잡한 관계 분석 (소셜 네트워크, 추천 시스템)
- 엔티티 중심의 검색 (특정 인물, 기업, 제품)
- 다단계 추론이 필요한 질문


### 2.3 Vector RAG vs GraphRAG 비교

| 측면 | Vector RAG | GraphRAG (우리 시스템) |
|------|-----------|----------------------|
| **검색 방식** | 임베딩 유사도 | Cypher 쿼리 (그래프 패턴 매칭) |
| **관계 표현** | ❌ 불가능 | ✅ 명시적 관계 저장 |
| **복잡한 질의** | ❌ 제한적 | ✅ 다단계 관계 추론 가능 |
| **구현 난이도** | ⭐⭐ 쉬움 | ⭐⭐⭐⭐ 어려움 |
| **정확성** | 의미적 유사도 기반 | 구조적 관계 기반 (더 정확) |
| **예시 질문** | "AI 관련 뉴스 찾아줘" | "뉴스1의 경제 기사를 쓴 기자는?" |


#### Vector RAG의 한계를 GraphRAG가 해결하는 예시

**질문**: "뉴스1에서 발행한 경제 뉴스를 작성한 기자는 누구야?"

- **Vector RAG**: 
  - 문제: "뉴스1", "경제", "기자"를 모두 포함한 문서를 찾기 어려움
  - 결과: 부정확하거나 불완전한 답변

- **GraphRAG**: 
  - 해결: `(기자)-[:WRITTEN_BY]-(뉴스)-[:PUBLISHED_BY]-(뉴스1)` 경로 탐색
  - 결과: 정확한 관계 기반 답변

## 3. Neo4j를 활용한 GraphRAG 실습 준비작업



### 3.1 API Key 설정
- [OpenAI API Key 발급](https://platform.openai.com/api-keys)


In [1]:
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()


True

### 3.2 Neo4j 연결 클래스

In [2]:
class Singleton(type):
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super(Singleton, cls).__call__(*args, **kwargs)
        return cls._instances[cls]

In [3]:
# Neo4j 연결 클래스 (Singleton 패턴)
from neo4j import GraphDatabase

class Neo4jConnection(metaclass=Singleton):
    """Neo4j 데이터베이스 연결 및 작업을 위한 클래스"""
    
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        print("Neo4j 연결 성공!")
    
    def close(self):
        if self.driver is not None:
            self.driver.close()
            print("Neo4j 연결 종료")
    
    def execute_query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]


### 3.3 Neo4j 연결 및 데이터 확인 

In [4]:
# Neo4j 연결
neo4j_conn = Neo4jConnection(
    uri="bolt://localhost:7687",
    user="neo4j",
    password="test1234"
)


Neo4j 연결 성공!


In [5]:
# 저장된 데이터 확인
query = """
MATCH (n)
RETURN labels(n)[0] as NodeType, count(n) as Count
ORDER BY Count DESC
"""

results = neo4j_conn.execute_query(query)
print("Neo4j 데이터 현황:\n")
for record in results:
    print(f"  {record['NodeType']:<15} : {record['Count']:>3}개")


Neo4j 데이터 현황:

  Reporter        :  20개
  News            :  20개
  Publisher       :  15개
  Category        :   4개
  YearMonth       :   1개


## 4. GraphRAG 시스템 구축

### 4.1 LLM


In [61]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5-nano",
    reasoning_effort="high",        # 논리성 강화
)

### 4.2 질문 유형별 Cypher 쿼리 템플릿

GraphRAG의 핵심은 **자연어 질문을 Cypher 쿼리로 변환**하는 것입니다!

In [6]:
from enum import Enum

class CypherQueryTemplates(Enum):
    """다양한 질문 유형을 Enum으로 관리하는 Cypher 템플릿"""

    NEWS_BY_CATEGORY = "news_by_category"
    NEWS_BY_PUBLISHER = "news_by_publisher"
    NEWS_BY_REPORTER = "news_by_reporter"

    def build(self, **kwargs):
        """템플릿 유형에 따라 Cypher 쿼리 생성"""
        if self is CypherQueryTemplates.NEWS_BY_CATEGORY:
            category_name = kwargs["category_name"]
            limit_no = kwargs["limit_no"]
            return f"""
            MATCH (n:News)-[:BELONGS_TO]->(c:Category {{name: "{category_name}"}})
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   p.name as 언론사, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        if self is CypherQueryTemplates.NEWS_BY_PUBLISHER:
            publisher_name = kwargs["publisher_name"]
            limit_no = kwargs["limit_no"]
            return f"""
            MATCH (n:News)-[:PUBLISHED_BY]->(p:Publisher {{name: "{publisher_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        if self is CypherQueryTemplates.NEWS_BY_REPORTER:
            reporter_name = kwargs["reporter_name"]
            limit_no = kwargs["limit_no"]
            return f"""
            MATCH (n:News)-[:WRITTEN_BY]->(r:Reporter {{name: "{reporter_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, p.name as 언론사, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT {limit_no}
            """

        raise ValueError(f"지원하지 않는 템플릿: {self}")


#### 테스트 

In [7]:
# 카테고리로 뉴스 검색
query = CypherQueryTemplates.NEWS_BY_CATEGORY.build(category_name="경제", limit_no=3)
print(query)


            MATCH (n:News)-[:BELONGS_TO]->(c:Category {name: "경제"})
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   p.name as 언론사, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT 3
            


In [8]:
results = neo4j_conn.execute_query(query)

In [9]:
results

[<Record 제목='1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑' 내용="10월 세수 전년比 2.8조 늘어…법인세 7000억·소득세 9000억↑ 진도율 88.9%, 전년比 1.7%p↑…5년 평균과 유사 ⓒ News1 윤주희 디자이너 (세종=뉴스1) 임용우 기자 = 올해 1~10월 국세 수입이 지난해 같은 기간보다 37조 1000억 원 늘어난 것으로 집계됐다. 지난해와 올해 상반기 기업 실적 개선, 성과급 지급 확대 등이 이어지며 법인세와 소득세가 크게 증가한 영향이다. 기획재정부가 28일 발표한 '2025년 10월 국세수입 현황'에 따르면 10월 국세 수입은 41조 1000억 원으로 전년 동월(38조 3000억 원)보다 2조 8000억 원 증가했다. 법인세는 상반기 기업 실적 개선에 따른 중소기업 중간예납 분납분 증가, 이자·배당 원천소득 증가 등으로 7000억 원 늘었다. 소득세는 근로자 수 증가와 총급여 지급액 확대에 따른 근로소득세 증가로 9000억 원 증가했다. 부가가치세는 올해 2기 예정신고분 납부 증가와 환율 상승 영향 등으로 7000억 원 증" 언론사='뉴스1' 기자='임용우 기자' 발행일='2025-11-28 11:00:00' 뉴스링크='https://n.news.naver.com/mnews/article/421/0008630778'>,
 <Record 제목='지난 자율주행차 사고 47건…교통안전공단 통계 최초 공개' 내용="주행속도 시속 30㎞ 이하 저속 구간서 주로 발생 내년엔 '차량거동' 정보까지 확대 공개 자율주행자동차 사고통계 공개 누리집 모습.(한국교통안전공단 제공)뉴스1ⓒ news1 (서울=뉴스1) 김동규 기자 = 한국교통안전공단 자동차안전연구원(TS)이 정부와 함께 국내 자율주행차 사고통계를 처음으로 공개한다. 2022년 이후 사고가 매년 증가하는 가운데 자율주행모드 운행 중 사고가 전체의 35%에 이르는 것으로 나타나, 안전성 확보를 위한 체계적 데이터 공개가 본격화됐다는 평가다. TS 자

In [10]:
for record in results:
    print(f"제목: {record['제목']}")
    print(f"내용: {record['내용'][:50]}...")
    print(f"언론사: {record['언론사']}")
    print(f"뉴스링크: {record['뉴스링크']}")
    break 

제목: 1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑
내용: 10월 세수 전년比 2.8조 늘어…법인세 7000억·소득세 9000억↑ 진도율 88.9%,...
언론사: 뉴스1
뉴스링크: https://n.news.naver.com/mnews/article/421/0008630778


### 4.3 LangChain Tool 정의

이제 CypherQueryTemplates를 LangChain의 Tool로 변환하여 Agent가 사용할 수 있도록 만들겠습니다.


#### search_news_by_category

In [67]:
from langchain_core.tools import tool

@tool
def search_news_by_category(category_name:str, limit_no:int=2) -> str:
    """
    category_name(카테고리)에 속하는 뉴스 기사 찾는 Cypher 쿼리를 이용하여, 
    실제 Neo4j DB에서 실행한 결과를 imit_no개 반환하는 검색 도구입니다.
    """
    query = CypherQueryTemplates.NEWS_BY_CATEGORY.build(
        category_name=category_name, limit_no=limit_no)
    answers = neo4j_conn.execute_query(query)

    result = f"[{category_name}(카테고리)에 속하는 뉴스 기사들]\n"
    for answer in answers:
        result += f"""
        뉴스 제목: {answer['제목']}
            - 내용: {answer['내용']}
            - 언론사: {answer['언론사']}
            - 발행일: {answer['발행일']}
            - 뉴스링크: {answer['뉴스링크']}
        """

    return result

In [68]:
result = search_news_by_category.invoke({
    "category_name":"세계", "limit_no":2
})

print(result)

[세계(카테고리)에 속하는 뉴스 기사들]

        뉴스 제목: 파리 루브르박물관, 내년부터 비EU 관광객만 입장료 45% 인상
            - 내용: 22유로 → 32유로 27일(현지시간) 프랑스 파리 루브르 박물관 유리 피라미드 인근에 프랑스 CRS(공화국보안대) 폭동 진압 경찰들이 서 있다. 2025.10.27. ⓒ 로이터=뉴스 (서울=뉴스1) 신기림 기자 = 프랑스 파리 루브르 박물관이 비(非) 유럽연합(EU) 국적자의 경우 입장료를 45% 인상한다고 27일(현지시간) 밝혔다. AFP 통신에 따르면 내년 1월 14일부터 비EU 국적자의 루브르 박물관 입장료는 기존 22유로에서 32유로로 인상된다. EU 지역 방문객들의 입장료는 그대로 22유로다. 박물관 측은 비EU 외국인 입장료 인상이 박물관의 자금 개선에 기여해 연간 최대 2300만 달러 수익 목표 달성에 도움을 줄 것이라고 밝혔다. 하지만 프랑스 민주노동총연맹은 모든 국적에 균일하게 적용되던 입장료가 폐지되면 "차별"로 인식될 것이라며 경고했다. 박물관의 2024년 보고서에 따르면 지난해 전체 방문객은 870만 명으로 이 중 69%가 외국인이었다. 미국인이 가장 많았고 
            - 언론사: 뉴스1
            - 발행일: 2025-11-28 10:04:49
            - 뉴스링크: https://n.news.naver.com/mnews/article/421/0008630601
        
        뉴스 제목: 백악관 총격사건에 격분한 트럼프 ‘우려국’ 영주권 전면 재조사 지시
            - 내용: 美 이민국, 19개국 특정…아프간 등 대상 도널드 트럼프 미국 행정부가 반(反)이민 정책의 고삐를 죈다. 미 최대 명절인 추수감사절 전날 워싱턴DC 한복판에서 주방위군 겨냥 총격 사건이 발생하면서다. 총격 받은 주방위군 병사 2명 중 1명이 사망하는 등 사태가 악화하자 ‘미국 내 영주권 재조사’에도 속도가 붙을 전망이다. 조세프 에들로 미

#### search_news_by_publisher

In [69]:
@tool
def search_news_by_publisher(publisher_name:str, limit_no:int=2) -> str:
    """
    publisher_name(언론사)가 발생한 뉴스 기사 찾는 Cypher 쿼리를 이용하여,
    실제 Neo4j DB에서 실행한 결과를 imit_no개 반환하는 검색 도구입니다.
    """
    query = CypherQueryTemplates.NEWS_BY_PUBLISHER.build(
        publisher_name=publisher_name, limit_no=limit_no)
    answers = neo4j_conn.execute_query(query)

    result = f"[{publisher_name}(언론사)가 발생한 뉴스 기사들]\n"
    for answer in answers:
        result += f"""
        뉴스 제목: {answer['제목']}
            - 내용: {answer['내용']}
            - 카테고리: {answer['카테고리']}
            - 발행일: {answer['발행일']}
            - 뉴스링크: {answer['뉴스링크']}
        """

    return result

In [70]:
result = search_news_by_publisher.invoke({
    "publisher_name":"KBS", "limit_no":2
})

print(result)

[KBS(언론사)가 발생한 뉴스 기사들]

        뉴스 제목: ‘2026 부산 세계도서관정보대회’ 지원 국가위원회 출범
            - 내용: 내년 부산에서 열리는 세계도서관정보대회를 지원할 위원회가 본격적으로 가동됩니다. 문화체육관광부는 오늘(28일) 오전 국회 의원회관에서 출범식을 갖고, 한국도서관협회 소속 ‘2026 부산 세계도서관정보대회(WLIC) 국가위원회’를 공식 출범한다고 밝혔습니다. 국가위원회는 국회와 중앙·지방정부, 학계, 민간에서 활동하던 전문가 16명으로 꾸려졌으며, 이들은 내년 열릴 부산 세계도서관정보대회 준비와 정책 협력 역할을 수행할 예정입니다. 공동위원장에는 차지호 더불어민주당 의원과 정연욱 국민의힘 의원이, 부위원장에는 김희섭 국립중앙도서관장·이준승 부산시 행정부시장·이지연 연세대 문헌정보학과 교수·황정근 국회도서관장이 각각 위촉됐습니다. 세계도서관정보대회는 1928년부터 시작돼 내년 90회째이며, 세계 도서관과 관련 정보 분야 전문가들이 한자리에 모여 현안을 논의하고 도서관의 미래를 그려나가는 세계 최대 규모의 국제회의입니다. 우리나라에서 세계도서관정보대회가 열리는 건 2006년 서울 대회 이
            - 카테고리: 생활/문화
            - 발행일: 2025-11-28 08:57:21
            - 뉴스링크: https://n.news.naver.com/mnews/article/056/0012075511
        


#### search_news_by_reporter

In [71]:
@tool
def search_news_by_reporter(reporter_name:str, limit_no:int=2) -> str:
    """
    reporter_name(기자명)가 작성한 뉴스 기사 찾는 Cypher 쿼리를 이용하여, 
    실제 Neo4j DB에서 실행한 결과를 imit_no개 반환하는 검색 도구입니다.
    """
    query = CypherQueryTemplates.NEWS_BY_REPORTER.build(
        reporter_name=reporter_name, limit_no=limit_no)
    answers = neo4j_conn.execute_query(query)

    result = f"[{reporter_name}(기자명)가 작성한 뉴스 기사 작성한 뉴스 기사들]\n"
    for answer in answers:
        result += f"""
        뉴스 제목: {answer['제목']}
            - 내용: {answer['내용']}
            - 카테고리: {answer['카테고리']}
            - 언론사: {answer['언론사']}
            - 발행일: {answer['발행일']}
            - 뉴스링크: {answer['뉴스링크']}
        """

    return result

In [72]:
result = search_news_by_reporter.invoke({
    "reporter_name":"임용우 기자", "limit_no":2
})

print(result)

[임용우 기자(기자명)가 작성한 뉴스 기사 작성한 뉴스 기사들]

        뉴스 제목: 1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑
            - 내용: 10월 세수 전년比 2.8조 늘어…법인세 7000억·소득세 9000억↑ 진도율 88.9%, 전년比 1.7%p↑…5년 평균과 유사 ⓒ News1 윤주희 디자이너 (세종=뉴스1) 임용우 기자 = 올해 1~10월 국세 수입이 지난해 같은 기간보다 37조 1000억 원 늘어난 것으로 집계됐다. 지난해와 올해 상반기 기업 실적 개선, 성과급 지급 확대 등이 이어지며 법인세와 소득세가 크게 증가한 영향이다. 기획재정부가 28일 발표한 '2025년 10월 국세수입 현황'에 따르면 10월 국세 수입은 41조 1000억 원으로 전년 동월(38조 3000억 원)보다 2조 8000억 원 증가했다. 법인세는 상반기 기업 실적 개선에 따른 중소기업 중간예납 분납분 증가, 이자·배당 원천소득 증가 등으로 7000억 원 늘었다. 소득세는 근로자 수 증가와 총급여 지급액 확대에 따른 근로소득세 증가로 9000억 원 증가했다. 부가가치세는 올해 2기 예정신고분 납부 증가와 환율 상승 영향 등으로 7000억 원 증
            - 카테고리: 경제
            - 언론사: 뉴스1
            - 발행일: 2025-11-28 11:00:00
            - 뉴스링크: https://n.news.naver.com/mnews/article/421/0008630778
        


#### Text2CypherRetriever

In [ ]:
from neo4j_graphrag.retrievers import Text2CypherRetriever

@tool
def search_by_text2cypher(question:str, limit_no:int=2) -> str:
    """
    question(자연어 질의)를 Neo4j Text2Cypher 기반으로 분석하여 Cypher 쿼리를 자동 생성하고,
    실제 Neo4j DB에서 실행한 결과를 imit_no개 반환하는 검색 도구입니다.
    """
    
    # Neo4j 스키마 정의 (우리의 뉴스 데이터 구조)
    neo4j_schema = """
    노드 속성:
    News {title: STRING, content: STRING, link: STRING, publishDate: STRING}
    Category {name: STRING}  # 예: "경제", "IT/과학", "생활/문화", "세계"
    Publisher {name: STRING}  # 언론사명
    Reporter {name: STRING}   # 기자명
    YearMonth {period: STRING}  # 예: "2025-11"

    관계:
    (:News)-[:BELONGS_TO]->(:Category)
    (:News)-[:PUBLISHED_BY]->(:Publisher)
    (:News)-[:WRITTEN_BY]->(:Reporter)
    (:News)-[:PUBLISHED_IN]->(:YearMonth)
    (:Reporter)-[:WORKS_FOR]->(:Publisher)
    """

    # (선택사항) 예시 쿼리 제공 - LLM이 더 정확한 Cypher를 생성하도록 도움
    examples = [
        "사용자 질문: 'IT/과학 카테고리의 뉴스를 보여줘' "
        "쿼리: MATCH (n:News)-[:BELONGS_TO]->(c:Category {name: 'IT/과학'}) "
        "RETURN n.title LIMIT 5",
        
        "사용자 질문: '뉴스1에서 발행한 뉴스는?' "
        "쿼리: MATCH (n:News)-[:PUBLISHED_BY]->(p:Publisher {name: '뉴스1'}) "
        "RETURN n.title LIMIT 5"
    ]

    # Text2CypherRetriever 초기화
    retriever = Text2CypherRetriever(
        driver=neo4j_conn.driver,  # 이미 연결된 Neo4j driver 사용
        llm=model,
        neo4j_schema=neo4j_schema,
        examples=examples
    )

    answers = retriever.search(query_text=question)

    result = f"[문의 내용: {question}]\n[결과]\n"
    for item in answers.items[:limit_no]:
        result += f"""
            - {item.content}
        """

    return result

In [74]:
result = search_by_text2cypher.invoke({
    "question":"가장 많은 기사를 발행한 언론사는 어디야?"
})

In [75]:
print(result)

[문의 내용: 가장 많은 기사를 발행한 언론사는 어디야?]
[결과]

            - <Record publisher='뉴스1' articles=4>
        


### 4.4 모델에 도구 바인딩

In [76]:
tools = [
    search_news_by_category, search_news_by_publisher, 
    search_news_by_reporter, search_by_text2cypher
]

In [79]:
# 모델에 도구 바인딩 (모델이 도구를 사용할 수 있도록 연결)
model = model.bind_tools(tools)

print("도구 정의 및 바인딩 완료!")
print("\n사용 가능한 도구들:")
for tool in tools:
    print("=" * 50)
    print(f"Tool 이름: {tool.name}")
    print(f"Tool 설명: {tool.description}")
    print(f"Tool 인풋 파라미터: {tool.args}")
    print(f"Tool return_direct: {tool.return_direct}")

print(f"\n총 {len(tools)}개의 도구가 모델에 바인딩되었습니다.")

도구 정의 및 바인딩 완료!

사용 가능한 도구들:
Tool 이름: search_news_by_category
Tool 설명: category_name(카테고리)에 속하는 뉴스 기사 찾는 Cypher 쿼리를 이용하여, 
실제 Neo4j DB에서 실행한 결과를 imit_no개 반환하는 검색 도구입니다.
Tool 인풋 파라미터: {'category_name': {'title': 'Category Name', 'type': 'string'}, 'limit_no': {'default': 2, 'title': 'Limit No', 'type': 'integer'}}
Tool return_direct: False
Tool 이름: search_news_by_publisher
Tool 설명: publisher_name(언론사)가 발생한 뉴스 기사 찾는 Cypher 쿼리를 이용하여,
실제 Neo4j DB에서 실행한 결과를 imit_no개 반환하는 검색 도구입니다.
Tool 인풋 파라미터: {'publisher_name': {'title': 'Publisher Name', 'type': 'string'}, 'limit_no': {'default': 2, 'title': 'Limit No', 'type': 'integer'}}
Tool return_direct: False
Tool 이름: search_news_by_reporter
Tool 설명: reporter_name(기자명)가 작성한 뉴스 기사 찾는 Cypher 쿼리를 이용하여, 
실제 Neo4j DB에서 실행한 결과를 imit_no개 반환하는 검색 도구입니다.
Tool 인풋 파라미터: {'reporter_name': {'title': 'Reporter Name', 'type': 'string'}, 'limit_no': {'default': 2, 'title': 'Limit No', 'type': 'integer'}}
Tool return_direct: False
Tool 이름: search_by_text2cypher
Too

## 5. ReAct Agent with RAG

![img](https://substackcdn.com/image/fetch/$s_!jFpt!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F6b27a8a7-8f67-4558-a3f4-44bf512e6c92_1766x812.gif)

### 5.1 ReAct (Reasoning + Acting)란?

**ReAct**는 LLM이 **추론(Reasoning)**과 **행동(Acting)**을 결합하여 문제를 해결하는 패러다임입니다.

```
사용자 질문: "경제 카테고리의 뉴스를 찾아줘"
     ↓
┌────────────────────────────────────────────────┐
│  Thought (추론): 경제 카테고리 뉴스를 찾으려면  │
│  NEWS_BY_CATEGORY 템플릿을 사용해야겠다        │
└────────────────────────────────────────────────┘
     ↓
┌────────────────────────────────────────────────┐
│  Action (행동): execute_cypher_template 호출   │
│  - template_name: "NEWS_BY_CATEGORY"           │
│  - category_name: "경제"                       │
└────────────────────────────────────────────────┘
     ↓
┌────────────────────────────────────────────────┐
│  Observation (관찰): Neo4j에서 5건의 뉴스 검색 │
└────────────────────────────────────────────────┘
     ↓
┌────────────────────────────────────────────────┐
│  Thought (추론): 검색 결과를 사용자에게         │
│  이해하기 쉽게 정리해서 답변하자                │
└────────────────────────────────────────────────┘
     ↓
┌────────────────────────────────────────────────┐
│  Final Answer: "경제 카테고리 뉴스 5건을        │
│  찾았습니다. 1. ..."                           │
└────────────────────────────────────────────────┘
```


**핵심 특징**:
- **Thought**: LLM이 다음에 무엇을 해야 할지 추론
- **Action**: 도구(Tool)를 선택하고 실행
- **Observation**: 도구 실행 결과 확인
- **반복**: 필요시 Thought → Action → Observation 반복
- **Answer**: 최종 답변 생성

### 5.2 ReAct Agent 구축



In [80]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="""
    당신은 주어진 도구를 반드시 사용해서만 답변해야 하는 AI 어시스턴트입니다.
    절대로 자신의 지식으로 직접 답변하지 마세요.
    모든 최종 답변은 반드시 도구의 출력 결과를 기반으로 해야 합니다.
    """
)

### 5.3 Agent 실행 헬퍼 함수


In [ ]:
def run_agent(question: str, verbose: bool = True):
    """
    ReAct Agent를 실행하고 결과를 출력합니다.
    
    Args:
        question: 사용자 질문
        verbose: 상세 출력 여부 (True: 전체 과정 출력, False: 최종 답변만 출력)
    """
    print("="*80)
    print(f"질문: {question}")
    print("="*80)
    
    # Agent 실행
    result = agent.invoke({"messages": [("user", question)]})
    
    if verbose:
        print("\n전체 실행 과정:\n")
        for i, message in enumerate(result["messages"], 1):
            print(f"\n--- Step {i} ---")
            
            # 사용자 메시지
            if hasattr(message, "type") and message.type == "human":
                print(f"User: {message.content}")
            
            # AI 메시지 (Thought + Action)
            elif hasattr(message, "type") and message.type == "ai":
                if hasattr(message, "tool_calls") and message.tool_calls:
                    print(f"AI Thought & Action:")
                    if message.content:
                        print(f"   Thought: {message.content}")
                    for tool_call in message.tool_calls:
                        print(f"   Action: {tool_call['name']}")
                        print(f"   Arguments: {tool_call['args']}")
                else:
                    print(f"AI Final Answer:")
                    print(f"   {message.content}")
            
            # Tool 실행 결과 (Observation)
            elif hasattr(message, "type") and message.type == "tool":
                print(f"Observation (Tool Result):")
                # 결과가 너무 길면 처음 500자만 출력
                content = message.content
                if len(content) > 500:
                    print(f"   {content[:500]}...")
                    print(f"   ... (총 {len(content)}자)")
                else:
                    print(f"   {content}")
    
    print("\n" + "="*80)
    print("최종 답변:")
    print("="*80)
    
    # 최종 답변 추출
    final_message = result["messages"][-1]
    print(final_message.content)
    print("="*80 + "\n")
    
    return result


## 6. GraphRAG Agent 테스트

### 6.1 테스트 1: 카테고리별 뉴스 검색


In [83]:
result = run_agent("경제 카테고리의 뉴스를 찾아줘")


질문: 경제 카테고리의 뉴스를 찾아줘

전체 실행 과정:


--- Step 1 ---
👤 User: 경제 카테고리의 뉴스를 찾아줘

--- Step 2 ---
AI Thought & Action:
   Action: search_news_by_category
   Arguments: {'category_name': '경제', 'limit_no': 3}

--- Step 3 ---
Observation (Tool Result):
   [경제(카테고리)에 속하는 뉴스 기사들]

        뉴스 제목: 1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑
            - 내용: 10월 세수 전년比 2.8조 늘어…법인세 7000억·소득세 9000억↑ 진도율 88.9%, 전년比 1.7%p↑…5년 평균과 유사 ⓒ News1 윤주희 디자이너 (세종=뉴스1) 임용우 기자 = 올해 1~10월 국세 수입이 지난해 같은 기간보다 37조 1000억 원 늘어난 것으로 집계됐다. 지난해와 올해 상반기 기업 실적 개선, 성과급 지급 확대 등이 이어지며 법인세와 소득세가 크게 증가한 영향이다. 기획재정부가 28일 발표한 '2025년 10월 국세수입 현황'에 따르면 10월 국세 수입은 41조 1000억 원으로 전년 동월(38조 3000억 원)보다 2조 8000억 원 증가했다. 법인세는 상반기 기업 실적 개선에 따른 중소기업 중간예납 분납분 증가, 이자·배당 원천소득 증가 등으로 7000억 원 늘었다....
   ... (총 2172자)

--- Step 4 ---
AI Final Answer:
   다음 3건의 경제 카테고리 뉴스를 확인했습니다:

1) 1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑
- 언론사: 뉴스1
- 발행일: 2025-11-28 11:00:00
- 링크: https://n.news.naver.com/mnews/article/421/0008630778

2) 지난 자율주행차 사고 47건…교통안

In [84]:
print("최종 답변")
print(result['messages'][-1].content)

최종 답변
다음 3건의 경제 카테고리 뉴스를 확인했습니다:

1) 1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑
- 언론사: 뉴스1
- 발행일: 2025-11-28 11:00:00
- 링크: https://n.news.naver.com/mnews/article/421/0008630778

2) 지난 자율주행차 사고 47건…교통안전공단 통계 최초 공개
- 언론사: 뉴스1
- 발행일: 2025-11-28 13:47:35
- 링크: https://n.news.naver.com/mnews/article/421/0008631187

3) 아파트 보러 갔더니 “니하오”…외국인 집주인 56%가 중국인
- 언론사: 한겨레
- 발행일: 2025-11-28 09:00:08
- 링크: https://n.news.naver.com/mnews/article/028/0002778800

원하시면 기간이나 언론사 기준으로 더 많은 기사도 찾아드리겠습니다.


### 6.2 테스트 2: 키워드 검색


In [85]:
result = run_agent("자율주행에 관한 뉴스가 있어?")

질문: 자율주행에 관한 뉴스가 있어?

전체 실행 과정:


--- Step 1 ---
👤 User: 자율주행에 관한 뉴스가 있어?

--- Step 2 ---
AI Thought & Action:
   Action: search_news_by_category
   Arguments: {'category_name': '자율주행', 'limit_no': 3}

--- Step 3 ---
Observation (Tool Result):
   [자율주행(카테고리)에 속하는 뉴스 기사들]


--- Step 4 ---
AI Thought & Action:
   Action: search_news_by_category
   Arguments: {'category_name': '자율주행', 'limit_no': 3}

--- Step 5 ---
Observation (Tool Result):
   [자율주행(카테고리)에 속하는 뉴스 기사들]


--- Step 6 ---
AI Thought & Action:
   Action: search_by_text2cypher
   Arguments: {'question': '자율주행 최근 뉴스'}

--- Step 7 ---
Observation (Tool Result):
   [문의 내용: 자율주행 최근 뉴스]
[결과]


--- Step 8 ---
AI Thought & Action:
   Action: search_news_by_category
   Arguments: {'category_name': '자율주행', 'limit_no': 5}
   Action: search_by_text2cypher
   Arguments: {'question': '최근 자율주행 뉴스'}

--- Step 9 ---
Observation (Tool Result):
   [자율주행(카테고리)에 속하는 뉴스 기사들]


--- Step 10 ---
Observation (Tool Result):
   [문의 내용: 최근 자율주행 뉴스]
[결과]


In [86]:
print("최종 답변")
print(result['messages'][-1].content)

최종 답변
네, 자율주행 관련 뉴스가 검색되었습니다. 아래 기사 하나를 확인하실 수 있습니다.

- 제목: (페이지에서 정확한 제목 확인 필요, 인코딩으로 표기되어 읽기 어렵습니다)
- 링크: https://n.news.naver.com/mnews/article/421/0008631187
- 발행일: 2025-11-28 13:47:35
- 카테고리: 경제

추가로 자율주행 관련 기사들이 더 검색되었지만 제목이 인코딩되어 있어 읽기 어렵습니다. 원하시면 기간(예: 최근 일주일), 언론사, 또는 특정 키워드로 더 좁혀서 추가 기사들을 찾아드리겠습니다. 어떤 방식으로 더 찾아볼까요?


### 6.4 테스트 4: 기자 통계 조회


In [87]:
result = run_agent("기자별로 작성한 기사 수 통계를 보여줘")


질문: 기자별로 작성한 기사 수 통계를 보여줘

전체 실행 과정:


--- Step 1 ---
👤 User: 기자별로 작성한 기사 수 통계를 보여줘

--- Step 2 ---
AI Thought & Action:
   Action: search_by_text2cypher
   Arguments: {'question': '기자별로 작성한 기사 수를 보여줘.', 'limit_no': 50}

--- Step 3 ---
Observation (Tool Result):
   [문의 내용: 기자별로 작성한 기사 수를 보여줘.]
[결과]

            - <Record reporter='이광수 기자' article_count=1>
        
            - <Record reporter='장훈경 기자' article_count=1>
        
            - <Record reporter='정남구 기자' article_count=1>
        
            - <Record reporter='김동규 기자' article_count=1>
        
            - <Record reporter='임용우 기자' article_count=1>
        
            - <Record reporter='김종윤 기자' article_count=1>
        
            - <Record reporter='함영훈 기자' article_count=1>
        
   ...
   ... (총 1372자)

--- Step 4 ---
AI Final Answer:
   다음은 기자별 작성 기사 수 통계입니다 (총 기자 20명, 기사 수 합계 20건).

- 이광수 기자: 1건
- 장훈경 기자: 1건
- 정남구 기자: 1건
- 김동규 기자: 1건
- 임용우 기자: 1건
- 김종윤 기자: 1건
- 함영훈 기자: 1건
- 이정은 기자: 1건
- 김성식 기자: 1건
- 박상현 기자: 1건

In [88]:
print("최종 답변")
print(result['messages'][-1].content)

최종 답변
다음은 기자별 작성 기사 수 통계입니다 (총 기자 20명, 기사 수 합계 20건).

- 이광수 기자: 1건
- 장훈경 기자: 1건
- 정남구 기자: 1건
- 김동규 기자: 1건
- 임용우 기자: 1건
- 김종윤 기자: 1건
- 함영훈 기자: 1건
- 이정은 기자: 1건
- 김성식 기자: 1건
- 박상현 기자: 1건
- 이휘경 기자 ddehg@wowtv.co.kr: 1건
- 나확진 기자: 1건
- 안선제 기자: 1건
- 주원규 기자: 1건
- 노경조 기자: 1건
- 김소라 기자: 1건
- 정인균 기자: 1건
- 김진선 기자: 1건
- 김지헌 기자: 1건
- 신기림 기자: 1건

참고: 이휘경 기자 항목에 이메일이 함께 표시되어 있습니다. 표준화가 필요할 수 있습니다.


## 7. 리소스 정리


In [89]:
# Neo4j 연결 종료
neo4j_conn.close()


Neo4j 연결 종료
